# Cross-Subject Validation (8-Fold)
## EEG Channel Estimation — Reviewer Response

**Addresses:**
- Reviewer 1 Point 3: Cross-Subject Validation
- Reviewer 2 Point 2: Deep Learning Comparison (CNN, LSTM)
- Reviewer 2 Point 3: Sample Size (all 40 subjects)

**Scheme:**
- 8-fold cross-subject validation
- Each fold: 5 subjects as test set, 35 subjects as training set
- Models: ANN, Random Forest, BDTR, SVR (shallow ML) + CNN, LSTM (deep learning)
- Both input modes: original and derivative
- Within training set: all trials from 35 subjects pooled into one training set

**Prerequisites:**
- All 40 subjects must be completed (`training_cache_Asli_sub{id}.pkl` and `training_cache_Turunan_sub{id}.pkl`)
- `fixed_single_ann_params.json` must exist in OUTPUT_DIR


In [ ]:
# =============================================================================
# 0. IMPORTS & CONFIG
# =============================================================================

import numpy as np
import pandas as pd
import os
import pickle
import json
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── Sesuaikan path berikut ─────────────────────────────────────────────────────
OUTPUT_DIR   = '/content/drive/MyDrive/Thesis/channel_estimation'
DATA_DIR     = '/content/drive/MyDrive/Thesis/SAM40'

# ── Config ─────────────────────────────────────────────────────────────────────
N_FOLDS      = 8
N_TEST       = 5
ALL_SUBJECTS = list(range(1, 41))
TASKS        = ['Arithmetic', 'Mirror_image', 'Stroop']
N_TRIALS     = 3
MODES        = ['original', 'derivative']
WINDOW_SIZE  = 128
WEARABLE_CH  = ['Fp1', 'Fp2', 'F7', 'F8']

NEW_ESTIMATION_SCHEME = {
    'FC2': WEARABLE_CH,
    'FZ' : WEARABLE_CH,
    'CZ' : WEARABLE_CH,
    'CP2': WEARABLE_CH,
    'P4' : WEARABLE_CH,
}

CHANNEL_NAMES = [
    'CZ','FZ','Fp1','F7','F3','FC1','C3','FC5',
    'FT9','T7','CP5','CP1','P3','P7','PO9','O1',
    'PZ','OZ','O2','PO10','P8','P4','CP2','CP6',
    'T8','FT10','FC6','C4','FC2','F4','F8','Fp2'
]

FIXED_ANN_PARAMS_PATH = os.path.join(OUTPUT_DIR, 'fixed_single_ann_params.json')
CS_DIR = os.path.join(OUTPUT_DIR, 'cross_subject_results')
os.makedirs(CS_DIR, exist_ok=True)

print("Config loaded.")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"N_FOLDS    : {N_FOLDS} (x {N_TEST} test subjects each)")
print(f"Total subj : {len(ALL_SUBJECTS)}")

## 1. Helper Functions

In [ ]:
import scipy.io as sio

def load_mat(filepath):
    mat  = sio.loadmat(filepath)
    data = mat['Clean_data'].astype(np.float64)
    return pd.DataFrame(data.T, columns=CHANNEL_NAMES)

def normalize_data(train_data, test_data):
    scaler     = MinMaxScaler(feature_range=(-1, 1))
    train_norm = scaler.fit_transform(train_data)
    test_norm  = scaler.transform(test_data)
    return train_norm, test_norm, scaler

def compute_metrics(y_true, y_pred):
    r, _  = pearsonr(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((y_true - np.mean(y_true)) ** 2)
    nse   = 1 - (numer / denom) if denom != 0 else np.nan
    return {'R': round(r,4), 'RMSE': round(rmse,6),
            'MAE': round(mae,6), 'NSE': round(nse,4)}

def expand_features(df, wearable_chs, mode='original'):
    X = df[wearable_chs].copy()
    if mode == 'derivative':
        for ch in wearable_chs:
            grad = np.gradient(df[ch].values, 1/128)
            X[f'd_{ch}'] = grad
    return X, list(X.columns)

print("Helper functions defined.")

## 2. Create 8 Folds (Non-Overlapping, 5 Subjects per Fold)

In [ ]:
def create_folds(all_subjects, n_folds=8, n_test=5, seed=42):
    rng  = np.random.default_rng(seed)
    subs = np.array(all_subjects)
    rng.shuffle(subs)
    folds = []
    for i in range(n_folds):
        test_subs  = subs[i*n_test : (i+1)*n_test].tolist()
        train_subs = [s for s in subs if s not in test_subs]
        folds.append({'fold': i+1, 'train': train_subs, 'test': test_subs})
    return folds

folds = create_folds(ALL_SUBJECTS)

print("Fold assignments:")
for f in folds:
    print(f"  Fold {f['fold']}: test={f['test']} | "
          f"train ({len(f['train'])} subjects)")

## 3. Dataset Builder Functions

In [ ]:
def load_subject_data(subject_id, task, data_dir=DATA_DIR):
    """Load all trials for one subject and one task."""
    trials = []
    for trial in range(1, N_TRIALS + 1):
        fp = os.path.join(data_dir,
                          f"{task}_sub_{subject_id}_trial{trial}.mat")
        if not os.path.exists(fp):
            print(f"  [WARN] Not found: {fp}")
            return None
        trials.append(load_mat(fp))
    return trials

def build_pooled_training_set(train_subjects, task, input_mode):
    """Pool all trials from 35 train subjects into one training set."""
    X_list = []
    y_dict = {ch: [] for ch in NEW_ESTIMATION_SCHEME}

    for sub_id in train_subjects:
        trials = load_subject_data(sub_id, task)
        if trials is None:
            continue
        df_all = pd.concat(trials, axis=0).reset_index(drop=True)
        X_feat, _ = expand_features(df_all, WEARABLE_CH, mode=input_mode)
        X_list.append(X_feat.values)
        for ch in NEW_ESTIMATION_SCHEME:
            y_dict[ch].append(df_all[ch].values)

    if not X_list:
        return None, None

    X_pooled = np.vstack(X_list)
    y_pooled = {ch: np.concatenate(y_dict[ch]) for ch in NEW_ESTIMATION_SCHEME}
    return X_pooled, y_pooled

def build_test_set(test_subjects, task, input_mode):
    """Pool all trials from 5 test subjects."""
    X_list = []
    y_dict = {ch: [] for ch in NEW_ESTIMATION_SCHEME}

    for sub_id in test_subjects:
        trials = load_subject_data(sub_id, task)
        if trials is None:
            continue
        df_all = pd.concat(trials, axis=0).reset_index(drop=True)
        X_feat, _ = expand_features(df_all, WEARABLE_CH, mode=input_mode)
        X_list.append(X_feat.values)
        for ch in NEW_ESTIMATION_SCHEME:
            y_dict[ch].append(df_all[ch].values)

    if not X_list:
        return None, None

    X_test = np.vstack(X_list)
    y_test = {ch: np.concatenate(y_dict[ch]) for ch in NEW_ESTIMATION_SCHEME}
    return X_test, y_test

print("Dataset builder functions defined.")

## 4. Deep Learning Model Builders (CNN & LSTM)

In [ ]:
HALF_WIN = WINDOW_SIZE // 2

def make_windowed_dataset(X, y, window_size=WINDOW_SIZE):
    """Sliding window, stride=1, many-to-one (center timepoint)."""
    n_samples = X.shape[0]
    half      = window_size // 2
    X_win, y_win = [], []
    for i in range(half, n_samples - half):
        X_win.append(X[i - half : i + half])
        y_win.append(y[i])
    return np.array(X_win), np.array(y_win)

def build_cnn(input_shape):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(64,  kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(128, kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(64,  kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1)
    ], name='CNN_1D')
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse')
    return model

def build_lstm(input_shape):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(64,  return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='LSTM')
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse')
    return model

def train_eval_dl(model_fn, model_name, X_tr_n, y_tr_flat,
                   X_te_n, y_te_orig, sc_y):
    """Train and evaluate CNN or LSTM on windowed data."""
    start     = time.time()
    input_shape = (WINDOW_SIZE, X_tr_n.shape[1])

    X_tr_win, y_tr_win = make_windowed_dataset(X_tr_n, y_tr_flat)
    X_te_win, y_te_win = make_windowed_dataset(X_te_n, y_te_orig)

    scaler_win = MinMaxScaler(feature_range=(-1, 1))
    y_tr_win_n = scaler_win.fit_transform(
        y_tr_win.reshape(-1,1)).ravel()

    model = model_fn(input_shape)
    model.fit(
        X_tr_win, y_tr_win_n,
        epochs=100, batch_size=256,
        validation_split=0.1,
        callbacks=[keras.callbacks.EarlyStopping(
            patience=10, restore_best_weights=True)],
        verbose=0
    )

    y_pred_n = model.predict(X_te_win, verbose=0).ravel()
    y_pred   = scaler_win.inverse_transform(
                   y_pred_n.reshape(-1,1)).ravel()

    metrics = compute_metrics(y_te_win, y_pred)
    print(f"      {model_name:20s} R={metrics['R']:.4f} | "
          f"RMSE={metrics['RMSE']:.4f} | {time.time()-start:.1f}s")
    return metrics

print("DL model builders defined.")

## 5. Run One Fold (All Models)

In [ ]:
def run_fold(fold_info, input_mode, fixed_ann):
    fold_id    = fold_info['fold']
    train_subs = fold_info['train']
    test_subs  = fold_info['test']

    print(f"\n{'='*60}")
    print(f"  FOLD {fold_id} | Mode: {input_mode.upper()}")
    print(f"  Train: {len(train_subs)} subjects | Test: {test_subs}")
    print(f"{'='*60}")

    all_results = []

    for task in TASKS:
        print(f"\n  [Task: {task}]")

        X_train, y_train = build_pooled_training_set(
            train_subs, task, input_mode)
        X_test,  y_test  = build_test_set(
            test_subs,  task, input_mode)

        if X_train is None or X_test is None:
            print(f"  [SKIP] Incomplete data for {task}")
            continue

        for ch in NEW_ESTIMATION_SCHEME:
            print(f"\n    [{ch}]")

            y_tr = y_train[ch].reshape(-1, 1)
            y_te = y_test[ch].reshape(-1, 1)
            X_tr_n, X_te_n, _    = normalize_data(X_train, X_test)
            y_tr_n, y_te_n, sc_y = normalize_data(y_tr, y_te)
            y_tr_flat  = y_tr_n.ravel()
            y_te_orig  = sc_y.inverse_transform(y_te_n).ravel()

            # ── ANN (fixed params) ────────────────────────────────────────
            p    = fixed_ann[ch]
            arch = tuple(p['architecture'])
            lr   = p['learning_rate']
            ann  = keras.Sequential()
            ann.add(layers.Input(shape=(X_tr_n.shape[1],)))
            for units in arch:
                ann.add(layers.Dense(units, activation='tanh'))
            ann.add(layers.Dense(1))
            ann.compile(optimizer=keras.optimizers.Adam(lr), loss='mse')
            ann.fit(X_tr_n, y_tr_flat, epochs=200, batch_size=64,
                    validation_split=0.1,
                    callbacks=[keras.callbacks.EarlyStopping(
                        patience=15, restore_best_weights=True)],
                    verbose=0)

            # ── Shallow ML ────────────────────────────────────────────────
            shallow = {
                'ANN_MLP'      : ann,
                'Random_Forest': RandomForestRegressor(
                    n_estimators=100, random_state=42, n_jobs=-1),
                'BDTR'         : GradientBoostingRegressor(
                    n_estimators=100, learning_rate=0.15,
                    max_depth=3, random_state=42),
                'SVR'          : SVR(
                    kernel='rbf', C=10, epsilon=0.1, gamma=0.5),
            }

            for mn, model in shallow.items():
                try:
                    if mn != 'ANN_MLP':
                        model.fit(X_tr_n, y_tr_flat)
                    raw = (model.predict(X_te_n, verbose=0)
                           if mn == 'ANN_MLP'
                           else model.predict(X_te_n).reshape(-1,1))
                    y_pred  = sc_y.inverse_transform(
                                  raw.reshape(-1,1)).ravel()
                    metrics = compute_metrics(y_te_orig, y_pred)
                    all_results.append({
                        'Fold': fold_id, 'Task': task, 'Target_CH': ch,
                        'Input_Mode': input_mode, 'Model': mn,
                        'Model_Type': 'Shallow_ML', **metrics
                    })
                    print(f"      {mn:20s} R={metrics['R']:.4f}")
                except Exception as e:
                    print(f"      [ERROR] {mn}: {e}")

            # ── Deep Learning ─────────────────────────────────────────────
            for dl_name, dl_fn in [('CNN', build_cnn),
                                    ('LSTM', build_lstm)]:
                try:
                    metrics = train_eval_dl(
                        dl_fn, dl_name,
                        X_tr_n, y_tr_flat, X_te_n, y_te_orig, sc_y)
                    all_results.append({
                        'Fold': fold_id, 'Task': task, 'Target_CH': ch,
                        'Input_Mode': input_mode, 'Model': dl_name,
                        'Model_Type': 'Deep_Learning', **metrics
                    })
                except Exception as e:
                    print(f"      [ERROR] {dl_name}: {e}")

    df = pd.DataFrame(all_results)
    fold_path = os.path.join(
        CS_DIR, f"fold{fold_id}_{input_mode}.csv")
    df.to_csv(fold_path, index=False, encoding='utf-8-sig')
    print(f"\n  [SAVED] {fold_path}")
    return df

print("run_fold() defined.")

## 6. Execute All Folds

In [ ]:
# Load fixed ANN params
with open(FIXED_ANN_PARAMS_PATH, 'r') as f:
    fixed_ann = json.load(f)
print(f"Fixed ANN params loaded from {FIXED_ANN_PARAMS_PATH}")

all_results = []
start_total = time.time()

for mode in MODES:
    print(f"\n{'#'*60}")
    print(f"# MODE: {mode.upper()}")
    print(f"{'#'*60}")

    for fold in folds:
        # Skip if already done
        fold_path = os.path.join(CS_DIR, f"fold{fold['fold']}_{mode}.csv")
        if os.path.exists(fold_path):
            print(f"  [SKIP] Fold {fold['fold']} mode {mode} already done.")
            all_results.append(pd.read_csv(fold_path))
            continue

        result = run_fold(fold, mode, fixed_ann)
        if result is not None and len(result) > 0:
            all_results.append(result)

elapsed = time.time() - start_total
print(f"\nTotal elapsed: {elapsed/3600:.2f} hours")

## 7. Aggregate Results

In [ ]:
df_all = pd.concat(all_results, ignore_index=True)
df_all.to_csv(
    os.path.join(CS_DIR, 'cross_subject_all_results.csv'),
    index=False, encoding='utf-8-sig')

# Summary: mean ± std across 8 folds
def fmt(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

summary = (
    df_all.groupby(['Input_Mode', 'Model_Type', 'Model', 'Target_CH'])
    [['R','RMSE','MAE','NSE']]
    .agg(['mean','std'])
    .round(4)
    .reset_index()
)

# Format untuk display
summary_fmt = summary.copy()
for metric in ['R','RMSE','MAE','NSE']:
    summary_fmt[metric] = summary.apply(
        lambda r: fmt(r[(metric,'mean')], r[(metric,'std')]), axis=1)

display_cols = ['Input_Mode','Model_Type','Model','Target_CH',
                'R','RMSE','MAE','NSE']
print("\n=== CROSS-SUBJECT VALIDATION SUMMARY ===")
print(summary_fmt[display_cols].to_string(index=False))

summary_fmt[display_cols].to_csv(
    os.path.join(CS_DIR, 'cross_subject_summary_formatted.csv'),
    index=False, encoding='utf-8-sig')
print(f"\nSaved to {CS_DIR}")

## Notes

- Results are saved per fold to allow resuming if interrupted
- The skip logic checks if a fold CSV already exists before re-running
- CNN and LSTM use window_size=128 (1 second at fs=128 Hz), stride=1, many-to-one
- Shallow ML uses sample-level input (no windowing) for fair comparison within each fold
- Final summary reports mean ± std across 8 folds per model per channel
